In [1]:
!pip install -q peft

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import random
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, f1_score
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
# model to fine-tune
model_name = "unsloth/Llama-3.2-1B-Instruct"
device = "cuda"

# where the data files are
data_folder = "/kaggle/input/datasets/andrewsosaguaita/pubmedqa-rrf"

In [4]:
# load one split from its jsonl file
def load_split(file_name):
    questions = []
    abstracts = []
    answers = []

    with open(f"{data_folder}/{file_name}") as f:
        for line in f:
            entry = json.loads(line)
            questions.append(entry["question"])
            abstracts.append(entry["context"]["contexts"])
            answers.append(entry["final_decision"])

    return questions, abstracts, answers


train_questions, train_abstracts, train_answers = load_split("pubmedqa_train_rrf.jsonl")
val_questions, val_abstracts, val_answers = load_split("pubmedqa_val_rrf.jsonl")
test_questions, test_abstracts, test_answers = load_split("pubmedqa_test_rrf.jsonl")

print(len(train_questions), len(val_questions), len(test_questions))
print(len(train_abstracts[0]))

700 100 200
5


In [5]:
# load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32)
model = model.to(device)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [6]:
# build the message we send the model for one question
def build_prompt(question, abstracts):
    instructions = "Read the abstracts and answer the question. Answer with only one word: yes, no, or maybe."
    abstracts_text = "\n\n".join(abstracts)
    question_text = f"Abstracts:\n{abstracts_text}\n\nQuestion: {question}"

    conversation = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question_text},
    ]

    # put the conversation in llama's chat format
    prompt = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True, date_string="1 Jan 2026")
    return prompt


# check one prompt
prompt = build_prompt(train_questions[0], train_abstracts[0])
print(prompt[:600])
print("tokens:", len(tokenizer(prompt)["input_ids"]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 1 Jan 2026

Read the abstracts and answer the question. Answer with only one word: yes, no, or maybe.<|eot_id|><|start_header_id|>user<|end_header_id|>

Abstracts:
The purpose of this survey was to ascertain the most common surgical practices for attaining negative (tumor-free) surgical margins in patients desiring breast-conservation treatment for breast cancer to see if a consensus exists for optimal treatment of patients.We sent a survey to 1,000 surgeons interested in the treatmen
tokens: 1452


In [7]:
# the three answers the model can pick from
labels = ["yes", "no", "maybe"]

# get the token number for each answer
label_token_ids = []
for label in labels:
    token_id = tokenizer.encode(label, add_special_tokens=False)[0]
    label_token_ids.append(token_id)

print(label_token_ids)
for token_id in label_token_ids:
    print(repr(tokenizer.decode(token_id)))


# have the model pick yes, no, or maybe for one question
def pick_answer(which_model, question, abstracts):
    prompt = build_prompt(question, abstracts)
    prompt_tokens = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)

    # run the model once, only scoring the next word
    with torch.no_grad():
        output = which_model(**prompt_tokens, logits_to_keep=1)

    # the model's score for every possible next word
    next_word_scores = output.logits[0, -1]

    # grab the scores for just our three answers
    label_scores = []
    for token_id in label_token_ids:
        label_scores.append(next_word_scores[token_id].item())

    # keep the answer with the highest score
    best_index = label_scores.index(max(label_scores))
    return labels[best_index]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[9891, 2201, 37860]
'yes'
'no'
'maybe'


In [8]:
# try it on one test question
print("correct:", test_answers[0])
print("model:", pick_answer(model, test_questions[0], test_abstracts[0]))

correct: yes
model: no


In [9]:
# ask the model every test question
predictions_before = []
for i in range(len(test_questions)):
    question = test_questions[i]
    abstracts = test_abstracts[i]
    predictions_before.append(pick_answer(model, question, abstracts))

print("accuracy:", accuracy_score(test_answers, predictions_before))
print("macro f1:", f1_score(test_answers, predictions_before, average="macro"))
print("model:", Counter(predictions_before))
print("correct:", Counter(test_answers))

accuracy: 0.325
macro f1: 0.22957343370187408
model: Counter({'no': 143, 'maybe': 56, 'yes': 1})
correct: Counter({'yes': 103, 'no': 75, 'maybe': 22})


In [10]:
# the dumb baseline: always say yes
always_yes = ["yes"] * len(test_answers)

print("accuracy:", accuracy_score(test_answers, always_yes))
print("macro f1:", f1_score(test_answers, always_yes, average="macro", zero_division=0))

accuracy: 0.515
macro f1: 0.22662266226622663


In [11]:
# save memory by recomputing parts of the model during training
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# prompt tuning settings
prompt_config = PromptTuningConfig(
    task_type="CAUSAL_LM",
    num_virtual_tokens=20,
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text="Read the abstracts and answer the question with yes, no, or maybe.",
    tokenizer_name_or_path=model_name,
)

# freeze the model and add the soft prompt we'll train
prompt_model = get_peft_model(model, prompt_config)
prompt_model.print_trainable_parameters()

trainable params: 40,960 || all params: 1,235,855,360 || trainable%: 0.0033


In [12]:
# how wrong the model is on one training question
def get_loss(question, abstracts, answer):
    prompt = build_prompt(question, abstracts)
    prompt_tokens = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)

    # run the model with the soft prompt added, only scoring the next word
    output = prompt_model(**prompt_tokens, logits_to_keep=1)
    next_word_scores = output.logits[0, -1]

    # token number of the correct answer
    correct_token_id = label_token_ids[labels.index(answer)]
    correct_token = torch.tensor(correct_token_id).to(device)

    # big when the model gives the right answer a low score
    loss = torch.nn.functional.cross_entropy(next_word_scores, correct_token)
    return loss


# try it on one training question
print(get_loss(train_questions[0], train_abstracts[0], train_answers[0]))

tensor(2.2489, device='cuda:0', grad_fn=<NllLossBackward0>)


In [13]:
from peft import get_peft_model_state_dict, set_peft_model_state_dict

# same random shuffle every run
random.seed(42)

# questions per nudge, and how many rounds to train
group_size = 8
num_rounds = 3

# updates only the soft prompt, the rest of the model is frozen
optimizer = torch.optim.AdamW(prompt_model.parameters(), lr=3e-3)

In [14]:
# score a model on a set of questions
def score(which_model, questions, abstracts, answers):
    predictions = []
    for i in range(len(questions)):
        predictions.append(pick_answer(which_model, questions[i], abstracts[i]))

    accuracy = accuracy_score(answers, predictions)
    macro_f1 = f1_score(answers, predictions, average="macro")
    return accuracy, macro_f1, predictions

In [15]:
# train on all training questions once, in random order
def train_one_round():
    prompt_model.train()
    optimizer.zero_grad()

    order = list(range(len(train_questions)))
    random.shuffle(order)

    total_loss = 0
    for step in range(len(order)):
        i = order[step]
        loss = get_loss(train_questions[i], train_abstracts[i], train_answers[i])

        # add this question's share to the group
        (loss / group_size).backward()
        total_loss += loss.item()

        # nudge the soft prompt after every group
        if (step + 1) % group_size == 0:
            optimizer.step()
            optimizer.zero_grad()

        # check progress every 100 questions
        if (step + 1) % 100 == 0:
            print(step + 1, "average loss:", total_loss / (step + 1))

    return total_loss / len(order)

In [16]:
best_val_f1 = 0
best_round = 0
best_soft_prompt = None

for round_number in range(1, num_rounds + 1):
    print("round", round_number)
    train_loss = train_one_round()

    # check how it does on the val questions
    prompt_model.eval()
    val_accuracy, val_f1, val_predictions = score(prompt_model, val_questions, val_abstracts, val_answers)
    print("train loss:", train_loss, "| val accuracy:", val_accuracy, "| val macro f1:", val_f1)
    print("val answers:", Counter(val_predictions))

    # keep the soft prompt from the best round so far
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_round = round_number
        best_soft_prompt = get_peft_model_state_dict(prompt_model)
        prompt_model.save_pretrained("/kaggle/working/ds266_soft_prompt")

print("best round:", best_round, "| val macro f1:", best_val_f1)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


round 1
100 average loss: 1.1637685081362725
200 average loss: 1.1164118242263794
300 average loss: 1.123746100018422
400 average loss: 1.0977217920869589
500 average loss: 1.066252045750618
600 average loss: 1.0552252969394127
700 average loss: 1.0330281317021166
train loss: 1.0330281317021166 | val accuracy: 0.6 | val macro f1: 0.3628756569212715
val answers: Counter({'yes': 80, 'no': 20})
round 2
100 average loss: 0.9164057892560958
200 average loss: 0.9136975264176727
300 average loss: 0.9416118834167719
400 average loss: 0.9357674908079207
500 average loss: 0.9274342629313469
600 average loss: 0.9204113280152281
700 average loss: 0.8953783163375089
train loss: 0.8953783163375089 | val accuracy: 0.51 | val macro f1: 0.48365471006980437
val answers: Counter({'maybe': 39, 'yes': 32, 'no': 29})
round 3
100 average loss: 0.9803216636553407
200 average loss: 0.9451770869083702
300 average loss: 0.9013077231869101
400 average loss: 0.9285990836191922
500 average loss: 1.0382822057902812


In [17]:
# load the soft prompt from the best round
set_peft_model_state_dict(prompt_model, best_soft_prompt)
prompt_model.eval()

# score the trained model on the test questions
test_accuracy, test_f1, predictions_after = score(prompt_model, test_questions, test_abstracts, test_answers)

print("best round:", best_round)
print("accuracy:", test_accuracy)
print("macro f1:", test_f1)
print("model:", Counter(predictions_after))
print("correct:", Counter(test_answers))

best round: 2
accuracy: 0.47
macro f1: 0.4362279117662336
model: Counter({'no': 78, 'maybe': 70, 'yes': 52})
correct: Counter({'yes': 103, 'no': 75, 'maybe': 22})
